# Retail E-Commerce BI Analytics
## Data Quality & Anomaly Audit

This notebook audits the Olist source tables before KPI publication. It checks structural integrity, missingness, value domains, temporal consistency, referential integrity, and reconciliation across analytical grains.

**Important:** unusual records are not automatically deleted. Each finding must be classified as an expected characteristic, a data-quality issue, or an analytical risk.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../data/raw')

orders = pd.read_csv(DATA_PATH / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_PATH / 'olist_customers_dataset.csv')
items = pd.read_csv(DATA_PATH / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_PATH / 'olist_products_dataset.csv')
payments = pd.read_csv(DATA_PATH / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_PATH / 'olist_order_reviews_dataset.csv')

## 1. Source-table dimensions

These counts establish the expected starting point for the audit.

In [ ]:
tables = {
    'orders': orders,
    'customers': customers,
    'order_items': items,
    'products': products,
    'order_payments': payments,
    'order_reviews': reviews
}

pd.DataFrame({name: {'rows': len(df), 'columns': len(df.columns)} for name, df in tables.items()}).T

## 2. Primary-key uniqueness

A duplicate key is only a defect when uniqueness is expected at that table's grain. Child tables are intentionally excluded from blanket uniqueness checks.

In [ ]:
key_checks = {
    'orders.order_id': orders['order_id'].duplicated().sum(),
    'customers.customer_id': customers['customer_id'].duplicated().sum(),
    'products.product_id': products['product_id'].duplicated().sum(),
}
pd.Series(key_checks, name='duplicate_rows')

## 3. Referential-integrity checks

Identify child records whose parent key is absent.

In [ ]:
referential_checks = {
    'items.order_id_missing': (~items['order_id'].isin(orders['order_id'])).sum(),
    'items.product_id_missing': (~items['product_id'].isin(products['product_id'])).sum(),
    'orders.customer_id_missing': (~orders['customer_id'].isin(customers['customer_id'])).sum(),
    'payments.order_id_missing': (~payments['order_id'].isin(orders['order_id'])).sum(),
    'reviews.order_id_missing': (~reviews['order_id'].isin(orders['order_id'])).sum(),
}
pd.Series(referential_checks, name='orphan_rows')

## 4. Missingness profile

Missing values are reported by table and field. Missing delivery timestamps and missing reviews can be legitimate, so they are investigated rather than automatically imputed or deleted.

In [ ]:
missingness = []
for name, df in tables.items():
    for column in df.columns:
        missing = df[column].isna().sum()
        if missing:
            missingness.append({
                'table': name,
                'column': column,
                'missing_rows': missing,
                'missing_pct': round(100 * missing / len(df), 2)
            })

missingness_df = pd.DataFrame(missingness).sort_values(['table', 'missing_pct'], ascending=[True, False])
missingness_df

## 5. Value-domain anomalies

These checks flag values outside the expected business domain. They do not decide whether every flagged row should be removed.

In [ ]:
value_checks = {
    'negative_item_price': (items['price'] < 0).sum(),
    'zero_item_price': (items['price'] == 0).sum(),
    'negative_freight': (items['freight_value'] < 0).sum(),
    'negative_payment': (payments['payment_value'] < 0).sum(),
    'zero_payment': (payments['payment_value'] == 0).sum(),
    'invalid_review_score': ((reviews['review_score'] < 1) | (reviews['review_score'] > 5)).sum(),
    'negative_installments': (payments['payment_installments'] < 0).sum(),
}
pd.Series(value_checks, name='issue_count')

## 6. Temporal consistency

Check whether operational timestamps violate the expected sequence.

In [ ]:
temporal_checks = {
    'approval_before_purchase': ((orders['order_approved_at'].notna()) &
                                  (orders['order_approved_at'] < orders['order_purchase_timestamp'])).sum(),
    'carrier_before_purchase': ((orders['order_delivered_carrier_date'].notna()) &
                                  (orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp'])).sum(),
    'delivery_before_purchase': ((orders['order_delivered_customer_date'].notna()) &
                                   (orders['order_delivered_customer_date'] < orders['order_purchase_timestamp'])).sum(),
    'delivery_before_carrier': ((orders['order_delivered_customer_date'].notna()) &
                                 (orders['order_delivered_carrier_date'].notna()) &
                                 (orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date'])).sum(),
}
pd.Series(temporal_checks, name='issue_count')

## 7. Delivery anomaly distribution

Delivery duration is calculated only where both purchase and customer-delivery timestamps exist. Extreme values are inspected rather than removed solely because they are large.

In [ ]:
orders['delivery_days_audit'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

orders['delivery_days_audit'].describe(percentiles=[.01, .05, .5, .95, .99])

## 8. Order/payment/item reconciliation

Payment value and item value answer different questions. They should not be expected to match perfectly at the order level without considering freight, vouchers, installments, and the source's payment semantics.

The key requirement is that neither measure is accidentally multiplied by child-table joins.

In [ ]:
order_item_value = items.groupby('order_id', as_index=False).agg(
    merchandise_value=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    item_count=('order_item_id', 'count')
)

order_payment_value = payments.groupby('order_id', as_index=False).agg(
    payment_value=('payment_value', 'sum'),
    payment_records=('payment_sequential', 'count')
)

reconciliation = order_item_value.merge(order_payment_value, on='order_id', how='outer', indicator=True)
reconciliation['_abs_difference'] = (reconciliation['merchandise_value'] + reconciliation['freight_value'] - reconciliation['payment_value']).abs()
reconciliation['_abs_difference'].describe(percentiles=[.5, .9, .95, .99])

## 9. Join-multiplication test

A deliberately broad join is shown only as a diagnostic. It must not be used as the universal KPI table.

In [ ]:
master = (orders[['order_id', 'customer_id']]
          .merge(items[['order_id', 'product_id', 'price']], on='order_id', how='left')
          .merge(payments[['order_id', 'payment_value']], on='order_id', how='left')
          .merge(reviews[['order_id', 'review_id']], on='order_id', how='left'))

pd.Series({
    'source_orders': orders['order_id'].nunique(),
    'master_rows': len(master),
    'master_distinct_orders': master['order_id'].nunique(),
    'row_multiplier': round(len(master) / orders['order_id'].nunique(), 3)
})

## Audit conclusion

Use the results above to classify findings before changing data. The corrected analytical model should use:

- order grain for orders, AOV, delivery, lateness and customer-level metrics
- item grain for category/product sales and freight
- payment grain for payment mix and payment values
- review grain or an explicitly aggregated order-review grain for review analysis

The audit is successful when KPI definitions are reproducible and no metric depends on accidental row multiplication.